# Aggregated Microscopy & OD Analysis

This notebook combines single-cell microscopy measurements from three independent experiments (96-well, 24-well, and test-tube) into a single summary table. Per-condition means and standard deviations for cell length and area are computed, then merged with plate-reader OD data. A 2×2 facet plot summarizes the four key metrics (length, area, OD morning, OD afternoon) for each strain.

In [2]:
from pathlib import Path

import arcadia_pycolor as apc
import matplotlib.pyplot as plt
import pandas as pd

apc.mpl.setup()

## Data preparation

Measurements are loaded from three CSVs under `data/microscopy/`, one per experiment type. The `bead_present` column is coerced to boolean across all three datasets (the raw files use a mix of string representations). For the 96-well experiment, only the "no bead" and "4.5 mm bead" conditions are retained, and `volume_ml` is set to 1.0 since all wells share the same volume.

In [5]:
REPO_ROOT = Path().resolve().parent

MICROSCOPY_CSV_PATHS = {
    "96-well": REPO_ROOT / "data" / "microscopy" / "combined_dic_measurements_96well.csv",
    "24-well": REPO_ROOT / "data" / "microscopy" / "combined_dic_measurements_24well.csv",
    "ttubes": REPO_ROOT / "data" / "microscopy" / "combined_dic_measurements_ttubes.csv",
}
OD_DATA_PATH = REPO_ROOT / "data" / "plate-reader" / "Baseline_ODs_stdev.csv"
OUTPUT_CSV = REPO_ROOT / "data" / "aggregated_summary.csv"

BEAD_PRESENT_MAP = {
    "no": False,
    "0": False,
    "false": False,
    "1": True,
    "yes": True,
    "true": True,
}


def coerce_bead_present(series: pd.Series) -> pd.Series:
    """Normalize a bead_present column to boolean, handling mixed string formats."""
    return series.astype(str).str.lower().map(BEAD_PRESENT_MAP)

In [6]:
dfs = []
for experiment, csv_path in MICROSCOPY_CSV_PATHS.items():
    if not csv_path.exists():
        print(f"Warning: {csv_path} not found, skipping.")
        continue

    df = pd.read_csv(csv_path)
    df["experiment"] = experiment

    if experiment == "96-well":
        if "treatment" in df.columns:
            treatment_lower = df["treatment"].astype(str).str.lower()
            df["bead_present"] = treatment_lower.str.contains("4.5 mm bead")
            df = df[treatment_lower.isin(["no bead", "4.5 mm bead"])]
        elif "bead_present" in df.columns:
            df["bead_present"] = coerce_bead_present(df["bead_present"])
        df["volume_ml"] = 1.0
    else:
        if "bead_present" in df.columns:
            df["bead_present"] = coerce_bead_present(df["bead_present"])

    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
print(f"Combined: {len(combined):,} cells across {combined['experiment'].nunique()} experiments")
for exp, count in combined["experiment"].value_counts().items():
    print(f"  {exp}: {count:,}")

combined.sample(10, random_state=42).drop(columns="source_file", errors="ignore")

Combined: 90,835 cells across 3 experiments
  24-well: 42,648
  ttubes: 41,742
  96-well: 6,445


,axis_major_length,area,length,well_letter,well_num,strain,treatment,experiment,bead_present,volume_ml,volume_group
20406,8.320914,28.730000,8.320914,E,2,SP286,2 mL no bead,24-well,False,2.0,b
79356,11.073457,37.576094,11.073457,F,3,SP286,3 mL no bead,ttubes,False,3.0,c
72968,16.087825,60.127031,16.087825,E,9,SP286,4 mL bead,ttubes,True,4.0,d
45942,8.571597,31.581875,8.571597,F,9,SP286,4 mL bead,24-well,True,4.0,d
83472,11.519452,39.424531,11.519452,F,6,SP286,1 mL bead,ttubes,True,1.0,a
32962,7.885572,26.591094,7.885572,E,9,SP286,4 mL bead,24-well,True,4.0,d
19878,5.689572,19.699063,5.689572,E,2,SP286,2 mL no bead,24-well,False,2.0,b
42860,8.997350,32.215625,8.997350,F,7,SP286,2 mL bead,24-well,True,2.0,b
73986,9.406325,30.393594,9.406325,E,9,SP286,4 mL bead,ttubes,True,4.0,d
69927,9.994223,33.034219,9.994223,E,7,SP286,2 mL bead,ttubes,True,2.0,b


## Summary statistics

Cells are grouped by experiment, strain, bead condition, and volume. For each group, we compute mean and standard deviation of cell length and area, along with the number of cells measured.

In [ ]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """Compute per-condition summary statistics for length and area."""
    df = df.dropna(subset=["volume_ml"]).copy()
    if "strain" not in df.columns:
        df["strain"] = "unknown"

    group_cols = ["strain", "bead_present", "volume_ml"]
    if "experiment" in df.columns:
        group_cols.insert(0, "experiment")

    return (
        df.groupby(group_cols, observed=True)
        .agg(
            length_mean=("length", "mean"),
            length_stdev=("length", "std"),
            area_mean=("area", "mean"),
            area_stdev=("area", "std"),
            n_cells=("length", "count"),
        )
        .reset_index()
    )


summary = summarize(combined)
print(f"Summary: {len(summary)} condition groups")
summary.head(10)

## Merge with OD data

Baseline OD measurements (morning and afternoon, with standard deviations) are loaded from `data/plate-reader/Baseline_ODs_stdev.csv` and joined to the microscopy summary on experiment, strain, bead condition, and volume. The merged table is exported to `data/aggregated_summary.csv`.

In [ ]:
od_df = pd.read_csv(OD_PATH)
od_df.columns = od_df.columns.str.strip()
od_df = od_df.rename(columns={"beads": "bead_present", "volume": "volume_ml"})
od_df["bead_present"] = coerce_bead_present(od_df["bead_present"])
od_df["volume_ml"] = pd.to_numeric(od_df["volume_ml"], errors="coerce")

merge_keys = ["experiment", "strain", "bead_present", "volume_ml"]
summary = summary.merge(od_df, on=merge_keys, how="left")
summary.to_csv(OUTPUT_CSV, index=False)
print(f"Exported merged summary to {OUTPUT_CSV.relative_to(REPO_ROOT)}")

summary.head(10)

## Faceted comparison across experiments

`plot_facets` produces a 2×2 figure for a given strain, with one panel per metric: cell length, cell area, OD morning, and OD afternoon. Within each panel, error bars show ±1 SD, and separate lines distinguish experiment × bead-condition combinations across culture volumes.

In [ ]:
FACETS = [
    ("length_mean", "length", "length_stdev"),
    ("area_mean", "area", "area_stdev"),
    ("OD_morning_mean", "OD morning", "OD_morning_stdev"),
    ("OD_afternoon_mean", "OD afternoon", "OD_afternoon_stdev"),
]

BEAD_LABELS = {False: "no bead", True: "bead"}

X_VOLUMES = [1, 2, 3, 4, 5]


def plot_facets(df: pd.DataFrame, strain: str, output_path: Path):
    """2x2 facet plot of length, area, and OD metrics for one strain."""
    df["volume_ml"] = pd.to_numeric(df["volume_ml"], errors="coerce")
    df["bead_present"] = df["bead_present"].astype(bool)

    experiments = df["experiment"].unique() if "experiment" in df.columns else [""]
    df_strain = df[df["strain"] == strain]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), facecolor=apc.parchment)

    for idx, (mean_col, title, stdev_col) in enumerate(FACETS):
        ax = axes[idx // 2, idx % 2]
        for exp in experiments:
            for bead in [False, True]:
                subset = df_strain[
                    (df_strain["experiment"] == exp) & (df_strain["bead_present"] == bead)
                ]
                subset = subset[subset["volume_ml"].isin(X_VOLUMES)]
                if mean_col in subset.columns and stdev_col in subset.columns and not subset.empty:
                    ax.errorbar(
                        subset["volume_ml"],
                        subset[mean_col],
                        yerr=subset[stdev_col],
                        marker="o",
                        label=f"{exp} ({BEAD_LABELS[bead]})",
                        capsize=4,
                    )
        ax.set_title(title)
        ax.set_xlabel("Volume (mL)")
        ax.set_xticks(X_VOLUMES)
        ax.grid(True)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")
    fig.suptitle(f"Strain: {strain}")
    fig.tight_layout()

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches="tight", facecolor=apc.parchment)
    print(f"Saved: {output_path.relative_to(REPO_ROOT)}")

    return fig

### SP286

Length, area, and OD metrics across volumes and bead conditions for strain SP286.

In [ ]:
fig_sp286 = plot_facets(
    summary,
    strain="SP286",
    output_path=STRAIN_FIG_DIRS["SP286"] / "aggregated_summary_SP286.png",
)